<a href="https://colab.research.google.com/github/techasit239/DADS7202_PigPicture/blob/main/FreshCheck_Phase2_Foundation_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreshCheck Deep Lab — Phase 2 Foundation: CVAT Parser + Evaluator

**Step 1: Material-aware Segmentation — Shared utilities for all 3 experiments**

> 🔗 **ต้องรันก่อน Exp 1, 2, 3** — notebook นี้สร้าง ground truth masks จาก CVAT XML และเครื่องมือ eval ที่ทุก experiment ใช้ร่วมกัน

---

## Phase 2 Foundation ทำอะไร?

| ขั้น | งาน | Output |
|---|---|---|
| 2.0 | Setup + Drive mount | folders พร้อม |
| 2.1 | Parse CVAT XML → binary masks | `phase2/gt_masks/*.png` (150 ไฟล์) |
| 2.2 | Build test dataset DataFrame | `phase2/thai_test_set.csv` |
| 2.3 | Define IoU + Dice evaluation | `phase2/seg_metrics.py` |
| 2.4 | Visualization helpers | `phase2/seg_viz.py` |
| 2.5 | Sanity check + summary | print stats |

**⏱ เวลา: ~5 นาที** (ไม่ต้องใช้ GPU)

## โครงสร้างการทำงานของ Phase 2

```
Phase 2 Foundation  (notebook นี้)
       │
       ├──→  CVAT GT masks
       │
       ├──────────────────────┬──────────────────────┐
       │                      │                      │
   Exp 1: SAM 3          Exp 2: Florence-2     Exp 3: DINOv3
   + text prompt         + SAM 3 refine        + seg head (train)
       │                      │                      │
       └──────────────────────┴──────────────────────┘
                              │
                  เลือก best mask method → Phase 3
```

## ก่อนเริ่ม — ต้องมีอะไรบ้าง

ใน Drive ต้องมี:
- `FreshCheck/data/thai_retail/` — Thai pork loin images ครบ 150 รูป (ใช้ Master Filename Format)
- `FreshCheck/data/thai_retail_cvat_annotations.xml` — CVAT export (format "CVAT for images 1.1")

ถ้ายังไม่ครบ:
1. **ยังเก็บไม่ครบ 150 รูป?** — ทำ collection ตาม Data Collection Guide ก่อน
2. **CVAT polygon ยังไม่เสร็จ?** — ให้คนที่ทำ CVAT polygons export เป็น XML แล้วอัพขึ้น Drive


## 2.0 — Setup Environment

Mount Drive + define paths สำหรับทั้ง Phase 2 (3 experiments จะ import paths เดิมไปใช้)


In [ ]:
# 2.0.1 — Imports + setup
import os, json, random
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'NumPy:  {np.__version__}')
print(f'PIL:    {Image.__name__}')


In [ ]:
# 2.0.2 — Mount Drive + define paths (shared across all Phase 2 notebooks)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/FreshCheck'

# Inputs
THAI_DATA_DIR = f'{PROJECT_ROOT}/data/thai_retail'
CVAT_XML_PATH = f'{PROJECT_ROOT}/data/thai_retail_cvat_annotations.xml'

# Phase 2 outputs
PHASE2_DIR     = f'{PROJECT_ROOT}/phase2'
GT_MASKS_DIR   = f'{PHASE2_DIR}/gt_masks'
EXP_RESULTS_DIR = f'{PHASE2_DIR}/exp_results'
CONFIGS_DIR    = f'{PROJECT_ROOT}/configs'
LOGS_DIR       = f'{PROJECT_ROOT}/logs'

for d in [PHASE2_DIR, GT_MASKS_DIR, EXP_RESULTS_DIR, CONFIGS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

THAI_TEST_CSV = f'{PHASE2_DIR}/thai_test_set.csv'

# Validate inputs exist
assert os.path.exists(THAI_DATA_DIR), f'ไม่พบ {THAI_DATA_DIR} — ทีมต้องอัพรูป Thai retail ขึ้น Drive ก่อน'
assert os.path.exists(CVAT_XML_PATH), f'ไม่พบ {CVAT_XML_PATH} — Export CVAT เป็น XML แล้วอัพขึ้น Drive ก่อน'

print(f'[OK] Project root: {PROJECT_ROOT}')
print(f'     Thai data:    {THAI_DATA_DIR}')
print(f'     CVAT XML:     {CVAT_XML_PATH}')
print(f'\nPhase 2 outputs:')
print(f'     GT masks:     {GT_MASKS_DIR}')
print(f'     Test CSV:     {THAI_TEST_CSV}')


## 2.1 — Parse CVAT XML → Binary Ground Truth Masks

### ทำไมต้องแปลง XML → binary masks?

CVAT export polygon เป็น list of (x,y) points แต่โมเดล segmentation ต้องการ **binary mask** (รูปขนาดเดียวกับภาพต้นฉบับ ที่ pixel = 1 ที่เนื้อ, 0 ที่อื่น)

### CVAT for images 1.1 XML structure

```xml
<annotations>
  <image id="0" name="20260518_1000_FR_PK_P01.jpg" width="1024" height="768">
    <polygon label="meat_tissue" points="120.5,234.1;180.2,250.3;..."/>
    <polygon label="meat_tissue" points="..."/>  <!-- รูปอาจมี multiple regions -->
  </image>
  <image id="1" name="...">
    ...
  </image>
</annotations>
```

### Logic การแปลง
1. Parse XML → ดึง `<image>` ทุก element
2. สำหรับแต่ละ `<image>`: เริ่ม blank mask (`np.zeros(H, W)`)
3. วาด polygon ทุกอัน (ใช้ `PIL.ImageDraw.polygon`) → pixel ใน polygon = 255
4. Save เป็น PNG (binary mask)


In [ ]:
# 2.1.1 — CVAT XML parser
def parse_cvat_xml(xml_path):
    """
    Parse 'CVAT for images 1.1' XML format.
    Returns: list of dicts {filename, width, height, polygons[List[List[(x,y)]]]}
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    images = []
    for img_el in root.findall('image'):
        filename = img_el.get('name')
        # Filename อาจมี path prefix (เช่น 'images/foo.jpg') — เอาแค่ basename
        filename = os.path.basename(filename)
        width  = int(img_el.get('width'))
        height = int(img_el.get('height'))

        polygons = []
        for poly_el in img_el.findall('polygon'):
            # points format: 'x1,y1;x2,y2;...'
            pts_str = poly_el.get('points')
            pts = [tuple(map(float, p.split(','))) for p in pts_str.split(';')]
            polygons.append(pts)

        images.append({
            'filename': filename,
            'width': width,
            'height': height,
            'polygons': polygons,
            'n_polygons': len(polygons),
        })
    return images


def polygons_to_mask(polygons, height, width):
    """
    Convert list of polygons → binary mask (np.uint8 array, 0 or 255).
    Multiple polygons are unioned (filled).
    """
    mask = Image.new('L', (width, height), 0)
    drawer = ImageDraw.Draw(mask)
    for poly in polygons:
        # PIL.ImageDraw.polygon expects flat tuple of coords or list of (x,y) tuples
        drawer.polygon(poly, fill=255, outline=255)
    return np.array(mask, dtype=np.uint8)


# Test parser on the user's XML
parsed = parse_cvat_xml(CVAT_XML_PATH)
print(f'Parsed {len(parsed)} images from CVAT XML\n')
print('First 3 annotations:')
for img in parsed[:3]:
    print(f'  {img["filename"]:<45} {img["width"]}×{img["height"]}  '
          f'{img["n_polygons"]} polygon(s)')

# Distribution
n_polys = [img['n_polygons'] for img in parsed]
print(f'\nPolygons per image:')
print(f'  min:  {min(n_polys)}')
print(f'  max:  {max(n_polys)}')
print(f'  mean: {np.mean(n_polys):.2f}')

# Sanity check: ถ้ามีรูปที่ไม่มี polygon เลย → flag
no_poly = [img['filename'] for img in parsed if img['n_polygons'] == 0]
if no_poly:
    print(f'\n[!] {len(no_poly)} รูปไม่มี polygon — ตรวจ CVAT annotations:')
    for f in no_poly[:5]:
        print(f'    {f}')


In [ ]:
# 2.1.2 — Generate binary masks for all images + save as PNG
print(f'Generating GT masks → {GT_MASKS_DIR}\n')

n_saved = 0
n_skipped = 0
for img in parsed:
    if img['n_polygons'] == 0:
        n_skipped += 1
        continue

    mask = polygons_to_mask(img['polygons'], img['height'], img['width'])
    mask_path = f'{GT_MASKS_DIR}/{Path(img["filename"]).stem}_mask.png'
    Image.fromarray(mask).save(mask_path)
    n_saved += 1

print(f'[OK] บันทึก {n_saved} masks')
if n_skipped > 0:
    print(f'[!]  ข้าม {n_skipped} รูปที่ไม่มี polygon')

# Verify a sample
sample = parsed[0]
sample_mask_path = f'{GT_MASKS_DIR}/{Path(sample["filename"]).stem}_mask.png'
sample_mask = np.array(Image.open(sample_mask_path))
foreground_pct = (sample_mask > 0).mean() * 100
print(f'\nSample mask check ({sample["filename"]}):')
print(f'  Shape: {sample_mask.shape}')
print(f'  Foreground (meat): {foreground_pct:.1f}% of pixels')


## 2.2 — Build Thai Test Set CSV

รวมข้อมูลของแต่ละรูป (image path + mask path + parsed Thai filename metadata) เป็น CSV ที่ทุก experiment โหลดได้


In [ ]:
# 2.2.1 — Import Thai filename parser from Phase 0
import sys
sys.path.insert(0, CONFIGS_DIR)

# Phase 0 v3 สร้าง parser ไว้ที่ configs/thai_filename_parser.py
parser_path = f'{CONFIGS_DIR}/thai_filename_parser.py'
assert os.path.exists(parser_path), f'ไม่พบ parser — รัน Phase 0 v3 ก่อน'

from thai_filename_parser import parse_thai_filename, is_valid_thai_filename
print('[OK] Loaded Thai filename parser from Phase 0')


In [ ]:
# 2.2.2 — Build dataset DataFrame
records = []
skipped = []

for img in parsed:
    fname = img['filename']
    img_path  = f'{THAI_DATA_DIR}/{fname}'
    mask_path = f'{GT_MASKS_DIR}/{Path(fname).stem}_mask.png'

    # Validate image file exists
    if not os.path.exists(img_path):
        skipped.append(f'{fname} (image missing)')
        continue
    if not os.path.exists(mask_path):
        skipped.append(f'{fname} (mask not generated — likely no polygon)')
        continue

    # Parse Thai filename → metadata
    try:
        meta = parse_thai_filename(fname)
    except ValueError as e:
        skipped.append(f'{fname} (invalid filename format)')
        continue

    records.append({
        'filename':    fname,
        'image_path':  img_path,
        'mask_path':   mask_path,
        'width':       img['width'],
        'height':      img['height'],
        'n_polygons':  img['n_polygons'],
        'class':       meta['class'],
        'class_code':  meta['class_code'],
        'source':      meta['source'],
        'source_code': meta['source_code'],
        'piece_id':    meta['piece_id'],
    })

df = pd.DataFrame(records)
df.to_csv(THAI_TEST_CSV, index=False)

print(f'[OK] Test set: {len(df)} samples')
print(f'     Saved to: {THAI_TEST_CSV}\n')

if skipped:
    print(f'[!] ข้าม {len(skipped)} รูป:')
    for s in skipped[:10]:
        print(f'    {s}')

print(f'\nDistribution:')
print(f'  Class:    ', df['class'].value_counts().to_dict())
print(f'  Source:   ', df['source'].value_counts().to_dict())
print(f'  Pieces:   {df["piece_id"].nunique()} unique pieces')


## 2.3 — Define IoU + Dice Evaluation

### Metrics ที่ใช้

**IoU (Intersection over Union)** = พื้นที่ทับซ้อน / พื้นที่รวม

```
IoU = |A ∩ B| / |A ∪ B|
```

**Dice Coefficient** (a.k.a F1 ของ pixel) = 2 × พื้นที่ทับซ้อน / (|A| + |B|)

```
Dice = 2|A ∩ B| / (|A| + |B|)
```

ค่าทั้งคู่อยู่ในช่วง 0-1:
- 1.0 = predicted mask ตรงกับ GT ทุก pixel
- 0.5 = ปกติของ segmentation task ที่ใช้ได้
- 0.0 = ไม่ทับซ้อนเลย

**ความต่าง:** Dice มักจะให้ค่าสูงกว่า IoU เล็กน้อย (Dice penalize false negative น้อยกว่า) — รายงานทั้งคู่เพื่อให้เห็นภาพชัด

### บันทึกเป็น module
Save เป็น `.py` file ให้ Exp 1/2/3 import ไปใช้ — ไม่ต้องเขียนซ้ำในทุก notebook


In [ ]:
# 2.3.1 — Save seg_metrics.py module
SEG_METRICS_CODE = '''"""
Segmentation metrics for FreshCheck Deep Lab Phase 2.
IoU + Dice for binary masks.
"""
import numpy as np


def binarize(mask, threshold=0.5):
    """Convert mask to binary 0/1. Accepts uint8 (0-255) or float (0-1)."""
    arr = np.asarray(mask)
    if arr.dtype == np.uint8 and arr.max() > 1:
        return (arr >= threshold * 255).astype(np.uint8)
    return (arr >= threshold).astype(np.uint8)


def compute_iou(pred_mask, gt_mask, threshold=0.5):
    """
    Compute IoU between predicted mask and ground truth.

    Args:
        pred_mask: predicted mask (HxW), uint8 or float
        gt_mask:   ground truth mask (HxW), uint8 or float
        threshold: threshold to binarize predictions

    Returns:
        iou: float in [0, 1]
    """
    pred = binarize(pred_mask, threshold)
    gt   = binarize(gt_mask, threshold)

    intersection = np.logical_and(pred, gt).sum()
    union        = np.logical_or(pred, gt).sum()

    if union == 0:
        # Both masks empty → undefined; convention: 1.0 (perfect agreement on emptiness)
        return 1.0
    return float(intersection) / float(union)


def compute_dice(pred_mask, gt_mask, threshold=0.5):
    """
    Compute Dice coefficient between predicted mask and ground truth.
    Same args as compute_iou.
    """
    pred = binarize(pred_mask, threshold)
    gt   = binarize(gt_mask, threshold)

    intersection = np.logical_and(pred, gt).sum()
    sum_areas    = pred.sum() + gt.sum()

    if sum_areas == 0:
        return 1.0
    return float(2 * intersection) / float(sum_areas)


def evaluate_batch(pred_masks, gt_masks, threshold=0.5):
    """
    Compute IoU + Dice for a batch. Returns lists of per-sample metrics.

    Args:
        pred_masks: list of HxW arrays
        gt_masks:   list of HxW arrays (same length as pred_masks)

    Returns:
        dict with keys: iou_list, dice_list, mean_iou, mean_dice, std_iou, std_dice
    """
    assert len(pred_masks) == len(gt_masks), \
        f"Length mismatch: {len(pred_masks)} vs {len(gt_masks)}"

    ious  = [compute_iou(p, g, threshold) for p, g in zip(pred_masks, gt_masks)]
    dices = [compute_dice(p, g, threshold) for p, g in zip(pred_masks, gt_masks)]

    return {
        "iou_list":  ious,
        "dice_list": dices,
        "mean_iou":  float(np.mean(ious)),
        "mean_dice": float(np.mean(dices)),
        "std_iou":   float(np.std(ious)),
        "std_dice":  float(np.std(dices)),
        "n_samples": len(ious),
    }
'''

metrics_path = f'{CONFIGS_DIR}/seg_metrics.py'
with open(metrics_path, 'w') as f:
    f.write(SEG_METRICS_CODE)

print(f'[OK] Saved metrics module: {metrics_path}')


In [ ]:
# 2.3.2 — Test metrics
# Reload (avoid cached import)
import importlib
import seg_metrics
importlib.reload(seg_metrics)
from seg_metrics import compute_iou, compute_dice, evaluate_batch

# Sanity test 1: identical masks → IoU = Dice = 1.0
gt   = np.zeros((100, 100), dtype=np.uint8)
gt[20:80, 20:80] = 255
pred = gt.copy()
print(f'Identical masks: IoU={compute_iou(pred, gt):.3f}, Dice={compute_dice(pred, gt):.3f}')
assert compute_iou(pred, gt) == 1.0 and compute_dice(pred, gt) == 1.0

# Sanity test 2: zero overlap → IoU = Dice = 0
pred = np.zeros((100, 100), dtype=np.uint8)
pred[0:20, 0:20] = 255   # ไม่ทับซ้อนกับ gt
print(f'Zero overlap:    IoU={compute_iou(pred, gt):.3f}, Dice={compute_dice(pred, gt):.3f}')
assert compute_iou(pred, gt) == 0.0 and compute_dice(pred, gt) == 0.0

# Sanity test 3: 50% overlap
pred = np.zeros((100, 100), dtype=np.uint8)
pred[20:80, 50:80] = 255   # half overlap of gt[20:80, 20:80]
print(f'Half overlap:    IoU={compute_iou(pred, gt):.3f}, Dice={compute_dice(pred, gt):.3f}')

print('\n[OK] Metrics tests passed')


## 2.4 — Visualization Helper

Function ช่วย overlay predicted mask + GT mask บนรูปต้นฉบับเพื่อ debug


In [ ]:
# 2.4.1 — Save seg_viz.py module
SEG_VIZ_CODE = '''"""
Visualization helpers for FreshCheck Phase 2 segmentation experiments.
"""
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


def show_seg_comparison(image_path, gt_mask, pred_mask, title="", figsize=(15, 4)):
    """
    Display original image, GT mask, predicted mask, overlay difference.
    """
    img = np.array(Image.open(image_path).convert("RGB"))

    # Normalize masks to binary
    gt   = (np.asarray(gt_mask)   > 127).astype(np.uint8)
    pred = (np.asarray(pred_mask) > 127).astype(np.uint8)

    # Difference overlay: green=TP, red=FP, blue=FN
    diff = np.zeros_like(img)
    tp = (pred == 1) & (gt == 1)
    fp = (pred == 1) & (gt == 0)
    fn = (pred == 0) & (gt == 1)
    diff[tp] = [0, 200, 0]    # green = correct
    diff[fp] = [200, 0, 0]    # red   = false positive
    diff[fn] = [0, 0, 200]    # blue  = false negative
    overlay = (img * 0.5 + diff * 0.5).astype(np.uint8)

    fig, axes = plt.subplots(1, 4, figsize=figsize)
    axes[0].imshow(img);     axes[0].set_title("Image");       axes[0].axis("off")
    axes[1].imshow(gt, cmap="gray");   axes[1].set_title("GT mask");     axes[1].axis("off")
    axes[2].imshow(pred, cmap="gray"); axes[2].set_title("Predicted");   axes[2].axis("off")
    axes[3].imshow(overlay)
    axes[3].set_title("TP=green FP=red FN=blue")
    axes[3].axis("off")
    if title:
        fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    return fig


def plot_iou_distribution(results_dict, figsize=(12, 4)):
    """
    Plot histogram of IoU + Dice for an experiment result.

    Args:
        results_dict: dict from evaluate_batch (has "iou_list", "dice_list")
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    axes[0].hist(results_dict["iou_list"], bins=20, color="#1e3a5f", alpha=0.8)
    axes[0].axvline(results_dict["mean_iou"], color="#c4583a", linestyle="--",
                    label=f'mean={results_dict["mean_iou"]:.3f}')
    axes[0].set_xlabel("IoU"); axes[0].set_ylabel("Count")
    axes[0].set_title("IoU distribution"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].hist(results_dict["dice_list"], bins=20, color="#4a7c59", alpha=0.8)
    axes[1].axvline(results_dict["mean_dice"], color="#c4583a", linestyle="--",
                    label=f'mean={results_dict["mean_dice"]:.3f}')
    axes[1].set_xlabel("Dice"); axes[1].set_ylabel("Count")
    axes[1].set_title("Dice distribution"); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    return fig
'''

viz_path = f'{CONFIGS_DIR}/seg_viz.py'
with open(viz_path, 'w') as f:
    f.write(SEG_VIZ_CODE)

print(f'[OK] Saved viz module: {viz_path}')


## 2.5 — Sanity Check + Summary

แสดงตัวอย่าง GT mask 4 รูป (1 รูปต่อ class) เพื่อยืนยันว่า CVAT polygon แปลงถูกต้อง


In [ ]:
# 2.5.1 — Display sample images + GT masks for verification
fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for row, cls in enumerate(['Fresh', 'Half-Fresh', 'Spoiled']):
    subset = df[df['class'] == cls]
    if len(subset) == 0:
        continue
    sample = subset.iloc[0]

    img  = Image.open(sample['image_path']).convert('RGB')
    mask = Image.open(sample['mask_path']).convert('L')

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'{cls}: {sample["filename"][:30]}', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(mask, cmap='gray')
    fg_pct = (np.array(mask) > 0).mean() * 100
    axes[row, 1].set_title(f'GT mask ({fg_pct:.1f}% foreground)', fontsize=10)
    axes[row, 1].axis('off')

plt.suptitle('Sample Images + CVAT Ground Truth Masks', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# 2.5.2 — Final summary
print('=' * 60)
print('PHASE 2 FOUNDATION — Complete')
print('=' * 60)
print(f'\nProject root: {PROJECT_ROOT}\n')

outputs = [
    (THAI_TEST_CSV, 'Thai test set CSV'),
    (f'{CONFIGS_DIR}/seg_metrics.py', 'IoU + Dice metrics module'),
    (f'{CONFIGS_DIR}/seg_viz.py', 'Visualization helpers'),
    (GT_MASKS_DIR, f'GT masks directory ({len(df)} masks)'),
]

for path, desc in outputs:
    if os.path.exists(path):
        if os.path.isdir(path):
            n_files = len(list(Path(path).glob('*.png')))
            print(f'[OK] {desc:<35} → {n_files} files')
        else:
            sz = os.path.getsize(path) / 1024
            print(f'[OK] {desc:<35} → {sz:.1f} KB')

print(f'\nThai test set: {len(df)} samples ready')
print(f'  Pieces: {df["piece_id"].nunique()}')
print(f'  Per class: {df["class"].value_counts().to_dict()}')

print(f'\nNext steps:')
print(f'  → Run Exp 1: SAM 3 + text prompt')
print(f'  → Run Exp 2: Florence-2 → SAM 3')
print(f'  → Run Exp 3: DINOv3 + segmentation head')
print(f'  → ทุก experiment โหลด {THAI_TEST_CSV} + import seg_metrics')
